# AI1: Thí nghiệm Tiếp tục Huấn luyện (Continuation) Silver v1 trên Kaggle

### Tuyên bố quan trọng & Phạm vi thực nghiệm
1. **Bản chất dữ liệu**: 56 cặp dữ liệu silver là tập dữ liệu huấn luyện thực nghiệm do trợ lý AI rà soát và tuyển chọn, **KHÔNG PHẢI nhãn gold do chuyên gia con người thẩm định** (not expert or human gold).
2. **Không thay thế production**: Thực nghiệm này nhằm đo lường chẩn đoán tác động của 56 cặp xếp hạng hỗ trợ (support-ranking) trong điều kiện kiểm soát. Kết quả **không cấu thành việc phê duyệt đưa vào môi trường thực tế (production)**.
3. **Thí nghiệm mới (Continuation)**: Do bundle 1.233 so sánh huấn luyện ban đầu trong quá khứ không còn lưu giữ đầy đủ, đây là **thực nghiệm tiếp tục mới** được dựng từ các cặp so sánh trong tập `train` của `semantic.jsonl` và 56 cặp silver thực nghiệm, không phải tái hiện tuyệt đối 1.233 nhóm cũ.
4. **Không tự động upload/deploy**: Không tự động gửi dữ liệu ra bên ngoài hoặc kích hoạt dịch vụ production.

### Thiết kế 3 Nhánh (Arms A/B/C)
- **Nhánh A (Baseline M1)**: Giữ nguyên checkpoint M1 retention cũ (step 0), không fine-tune.
- **Nhánh B (Control)**: Tiếp tục huấn luyện chỉ với dữ liệu train cũ của `semantic.jsonl`.
- **Nhánh C (Silver Experiment)**: Huấn luyện với dữ liệu train cũ + 56 cặp silver thực nghiệm theo tỷ lệ cố định (25% slots silver, lấy mẫu có hoàn lại).
- **Quy tắc công bằng**: Nhánh B và C có cùng ngân sách 155 update steps, batch size 8, learning rate 2e-5, hinge margin 0.1, retention loss với frozen teacher M1 và cùng random seed 42.
- **Quy tắc chọn model**: Chọn checkpoint (hoặc baseline step 0) duy nhất dựa trên **Validation Job-Macro MRR** (ties ưu tiên checkpoint sớm hơn). Tập **Test hoàn toàn độc lập và chỉ được đánh giá sau khi hoàn tất lựa chọn**.

## 1. Khám phá Bundle và Cấu hình Đường dẫn

Quy trình tự động tìm kiếm thư mục bundle chứa `manifest.json` trong `/kaggle/input` (hoặc cấu hình tùy chỉnh qua `BUNDLE_ROOT`), đảm bảo không phỏng đoán ngẫu nhiên file.

In [ ]:
import os
import sys
from pathlib import Path

# Cấu hình đường dẫn gốc bundle (ưu tiên biến môi trường hoặc tìm trong /kaggle/input)
def discover_bundle_root():
    env_root = os.environ.get("BUNDLE_ROOT")
    if env_root and Path(env_root).exists() and (Path(env_root) / "manifest.json").exists():
        return Path(env_root).resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates = [p for p in kaggle_input.rglob("manifest.json") if p.is_file()]
        if len(candidates) == 1:
            return candidates[0].parent.resolve()
        elif len(candidates) > 1:
            raise ValueError(f"Phat hien nhieu hon 1 manifest.json trong /kaggle/input: {candidates}")

    # Fallback cho moi truong cuc bo / test
    for candidate_name in ["ai1-kaggle-silver-v2", "ai1-kaggle-silver-v1"]:
        local_candidate = Path(f"./artifacts/{candidate_name}").resolve()
        if local_candidate.exists() and (local_candidate / "manifest.json").exists():
            return local_candidate

    raise FileNotFoundError("Khong tim thay bundle root hop le chua manifest.json. Vui long set BUNDLE_ROOT.")

BUNDLE_DIR = discover_bundle_root()
WORKING_DIR = Path("/kaggle/working").resolve() if Path("/kaggle/working").exists() else Path("./run_working").resolve()
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Thư mục Bundle xác định tại: {BUNDLE_DIR}")
print(f"Thư mục làm việc Working tại: {WORKING_DIR}")

## 2. Kiểm tra Môi trường GPU & Thư viện An toàn

Kiểm tra card GPU CUDA của Kaggle (T4 x2 hoặc P100), kiểm tra PyTorch và các thư viện cần thiết. Tránh cài đặt `pip install torch` làm ghi đè phiên bản GPU bằng CPU torch.

In [ ]:
import os
import subprocess
import sys

# Kiem tra torch GPU ma khong pha vo CUDA
try:
    import torch
    cuda_avail = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_avail else "CPU"
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA kha dung: {cuda_avail} ({device_name})")
    if not cuda_avail and not os.environ.get("FORCE_CPU_RUN"):
        print("CHU Y: CUDA hien chua kha dung. Hay bat GPU T4 hoac P100 trong Kaggle Accelerator.")
except ImportError:
    raise RuntimeError("Khong tim thay PyTorch. Hay su dung moi truong Kaggle GPU mac dinh, tranh chay pip install torch.")

# Kiem tra/cai dat transformers va safetensors ma tuyet doi khong ghi de torch
try:
    import transformers
    import safetensors
    print(f"Transformers: {transformers.__version__}, Safetensors: {safetensors.__version__}")
except ImportError:
    print("Dang cai dat transformers, safetensors (dung --no-deps de giu nguyen GPU torch)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "safetensors", "transformers"])

# Smoke check Tokenizer offline
from transformers import AutoTokenizer
model_dir = BUNDLE_DIR / "model"
if model_dir.exists():
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True, trust_remote_code=False)
    test_toks = tokenizer.encode("Kiem tra thu nghiem offline tokenizer", truncation=True, max_length=128)
    print(f"Tokenizer offline hoat dong chinh xac (token count: {len(test_toks)})")

## 3. Tách biệt Mã nguồn Tin cậy & Xác thực Toàn vẹn Manifest

Sao chép script runner tin cậy vào thư mục làm việc riêng biệt (`WORKING_DIR/runner`), sau đó kiểm tra checksum SHA-256 của toàn bộ bundle trước khi tiến hành.

In [ ]:
import hashlib
import json
import shutil
import sys
from pathlib import Path

# 1. XAC THUC MANIFEST TRUOC KHI IMPORT HOAC CHAY BAT KY CODE NAO
manifest_path = BUNDLE_DIR / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Khong tim thay manifest.json tai: {manifest_path}")

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest_info = json.load(f)

artifacts = manifest_info.get("artifacts", {})
if not artifacts:
    raise ValueError("Empty or invalid artifacts in manifest.json")

def compute_sha256(filepath: Path) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print(f"Bat dau xac thuc toan ven {len(artifacts)} artifacts theo manifest.json TRUOC KHI import...")
for rel_path, expected_sha in artifacts.items():
    p = BUNDLE_DIR / rel_path
    if not p.exists():
        raise FileNotFoundError(f"Thieu artifact duoc ghi trong manifest: {rel_path}")
    actual_sha = compute_sha256(p)
    if actual_sha != expected_sha:
        raise ValueError(f"SHA-256 mismatch cho artifact '{rel_path}': expected {expected_sha}, got {actual_sha}")

for p in BUNDLE_DIR.rglob("*"):
    if p.is_file():
        rel = str(p.relative_to(BUNDLE_DIR))
        if rel == "manifest.json" or rel.startswith("."):
            continue
        if rel not in artifacts:
            raise ValueError(f"Phat hien file bat thuong khong nam trong manifest: {rel}")

print(f"Toan ven 100%! Da xac thuc {len(artifacts)} artifacts an toan tuyet doi.")

# 2. Chi sao chep va import runner SAU KHI manifest xac thuc hop le
runner_src = BUNDLE_DIR / "runner" / "ai1_kaggle_experiment.py"
runner_dest_dir = WORKING_DIR / "runner"
runner_dest_dir.mkdir(parents=True, exist_ok=True)
runner_dest = runner_dest_dir / "ai1_kaggle_experiment.py"

if runner_src.exists():
    shutil.copy2(runner_src, runner_dest)
    print(f"Da sao chep runner script tin cay vao: {runner_dest}")
else:
    raise FileNotFoundError(f"Khong tim thay runner script tai: {runner_src}")

sys.path.insert(0, str(runner_dest_dir))
import ai1_kaggle_experiment
print("Da import thanh cong ai1_kaggle_experiment sau khi kiem tra manifest.")

## 4. Chạy Thực nghiệm Chính thức (Arms A, B, C)

Thực thi quy trình đầy đủ: 155 update steps, batch size 8, learning rate 2e-5, margin 0.1, retention loss với frozen teacher M1.
- Checkpoint selection được thực hiện độc lập trên tập **Validation**.
- Đánh giá cuối cùng trên tập **Test** với paired bootstrap CI 95% giữa Nhánh C và các nhánh A, B.

In [ ]:
output_experiment_dir = WORKING_DIR / "experiment_outputs"
output_experiment_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-I", str(runner_dest),
    "--semantic", str(BUNDLE_DIR / "data" / "semantic.jsonl"),
    "--silver", str(BUNDLE_DIR / "data" / "silver_comparisons.jsonl"),
    "--model", str(BUNDLE_DIR / "model"),
    "--output", str(output_experiment_dir),
    "--steps", "155",
    "--batch-size", "8",
    "--seed", "42",
    "--lr", "2e-5",
    "--margin", "0.1",
    "--retention-weight", "1.0",
    "--silver-ratio", "0.25",
]

print("Dang thuc thi tiep tuc huan luyen A/B/C:")
print(" ".join(cmd))
subprocess.check_call(cmd)
print("Thuc nghiem hoan thanh thanh cong!")

## 5. Tổng hợp Kết quả Thực nghiệm & Phân tích Ý nghĩa Thống kê

In [ ]:
import json

res_file = output_experiment_dir / "experiment_results.json"
with open(res_file, "r", encoding="utf-8") as f:
    results = json.load(f)

def fmt(v, spec=".4f"):
    if v is None:
        return "N/A"
    try:
        return format(v, spec)
    except Exception:
        return str(v)

print("=" * 70)
print("KET QUA SELECTION TRÊN TAP VALIDATION:")
print("=" * 70)
for arm_name, arm_data in results.get("arms", {}).items():
    step = arm_data.get("selected_step")
    val_mrr = arm_data.get("val_job_macro_mrr")
    print(f"{arm_name:6}: Selected Step = {step}, Val Job-Macro MRR = {fmt(val_mrr)}")

print()
print("=" * 70)
print("KET QUA DANH GIA TRÊN TAP TEST (Chon checkpoint truoc, test sau):")
print("=" * 70)
test_eval = results.get("test_evaluation", {})
for arm_key in ["arm_a", "arm_b", "arm_c"]:
    arm_eval = test_eval.get(arm_key, {})
    macro_mrr = arm_eval.get("macro_job", {}).get("mrr")
    macro_hit3 = arm_eval.get("macro_job", {}).get("hit3")
    print(f"{arm_key.upper():6}: Test Job-Macro MRR = {fmt(macro_mrr)}, Hit@3 = {fmt(macro_hit3)}")

bilingual = test_eval.get("arm_c", {}).get("bilingual", {})
if bilingual:
    print()
    print("Bilingual Breakdown Arm C (Test split):")
    print(f"- English MRR ({bilingual.get('en_count', 0)} queries): {fmt(bilingual.get('en_mrr'))}")
    print(f"- Vietnamese MRR ({bilingual.get('vi_count', 0)} queries): {fmt(bilingual.get('vi_mrr'))}")

boots = test_eval.get("bootstrap_comparisons", {})
boot_ca = boots.get("delta_c_minus_a", {})
boot_cb = boots.get("delta_c_minus_b", {})
print()
print("Paired Job Bootstrap 95% CI (1,000 resamples):")
print(f"- Delta (C - A): {fmt(boot_ca.get('mean_delta'), '+.4f')} (95% CI: [{fmt(boot_ca.get('ci_95_low'), '+.4f')}, {fmt(boot_ca.get('ci_95_high'), '+.4f')}])")
print(f"- Delta (C - B): {fmt(boot_cb.get('mean_delta'), '+.4f')} (95% CI: [{fmt(boot_cb.get('ci_95_low'), '+.4f')}, {fmt(boot_cb.get('ci_95_high'), '+.4f')}])")


## 6. Đóng gói Artifacts Kết quả để Tải về

Tất cả kết quả (metrics JSON, predictions JSONL của cả Arm A, B, C) được nén thành file zip trong `/kaggle/working` để tải về (file nhẹ < 1 MB, không chứa checkpoints nặng 2GB).

In [ ]:
import zipfile
from pathlib import Path

zip_out = WORKING_DIR / "ai1_experiment_results.zip"
print(f"Dang dong goi ket qua vao: {zip_out}...")

# Chi dong goi cac file ket qua metrics va predictions (A, B, C), bo qua checkpoints nang 2GB
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in output_experiment_dir.rglob("*"):
        if p.is_file() and "checkpoints" not in p.parts:
            rel = p.relative_to(output_experiment_dir)
            zf.write(p, arcname=str(rel))

print(f"Da dong goi thanh cong: {zip_out} ({zip_out.stat().st_size} bytes)")
print("File ket qua chi nang vai chuc KB (chua du predictions cua Arm A, B, C). Ban co the tai truc tiep tu muc Output.")
